In [5]:
from pydantic import BaseModel, Field
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langsmith import traceable, get_current_run_tree, Client
from operator import add
from typing import Any, Annotated, Dict, List
import yaml
from jinja2 import Template

from langchain_core.messages import BaseMessage, AIMessage, ToolMessage, HumanMessage, SystemMessage
from langchain_core.tools import BaseTool
from langchain_core.prompts import BasePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.prompts import StringPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import convert_to_openai_messages, convert_to_messages
from langchain_protocol import Literal

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document,Prefetch, FusionQuery
from qdrant_client import models

import instructor

import pandas as pd
import openai
import fastembed

from jinja2 import Template
from typing import List, Dict, Any, Optional, Union
from IPython.display import Image, display
from operator import add
from openai import OpenAI

import random
import ast
import inspect
import instructor
import json
import os
import importlib
import utils
from dotenv import load_dotenv
import numpy as np
load_dotenv()
importlib.reload(utils)

from utils import format_ai_message, parse_function_definition, get_type_from_annotation, parse_docstring_params, get_tool_descriptions

In [6]:
ls_client = Client()
ls_prompt = ls_client.pull_prompt("retrieval_generation_prompt")
ls_template = ls_prompt.messages[0].prompt.template

preprocessed_context = "- a \n - b"
question = "What is a?"

def build_prompt_with_jinja(preprocessed_context, question):
    jinja_template = """You are a helpful shopping assistant for answering questions about products in stock.
      You will be given a question and a list of context

      Instructions:
      - You need to answer the question based on the provided context only
      - Never use word context and refer to it as the available products
      - As an output you need to provide:

      * The answer to the question based on the provided context
      * The list of the IDs of the chuns that were used to answer the question.
      only return the ones that are used in the answer.
      * Short description (1-2 sentences) of the item based on the description provided in the context

      - The short description should have the name of the item.
      - The answer to the question should contain detailed information about the product and returned with
      detailed specification in bullet points.

      Context:
        {{preprocessed_context}}
      Question: 
        {{question}}
    """

    template = Template(jinja_template)
    rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)
    return rendered_template

def prompt_template_config(yaml_file, prompt_key):
    with open(yaml_file, 'r') as file:
        config = yaml.safe_load(file)

    prompt_entry = config['prompts'][prompt_key]
    template_content = prompt_entry['template'] if isinstance(prompt_entry, dict) else prompt_entry

    template = Template(template_content)

    return template


def prompt_template_registry(prompt_name):
    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template
    template = Template(template_content)
    return template


print(prompt_template_registry("retrieval_generation_prompt").render(preprocessed_context=preprocessed_context, question=question))

You are a helpful shopping assistant for answering questions about products in stock.
      You will be given a question and a list of context
      Instructions:
      - You need to answer the question based on the provided context only
      - Never use word context and refer to it as the available products
      - As an output you need to provide:
      * The answer to the question based on the provided context
      * The list of the IDs of the chuns that were used to answer the question.
      only return the ones that are used in the answer.
      * Short description (1-2 sentences) of the item based on the description provided in the context
      - The short description should have the name of the item.
      - The answer to the question should contain detailed information about the product and returned with
      detailed specification in bullet points.
      Context:
- a 
 - b
      Question: 
What is a?


In [7]:

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key,
)

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_69488/3853642596.py:18: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


Retrieve all item IDs from Amazon Items Qdrant Collection

In [17]:
import openai

# 1. Generate the embedding vector for your search term
response = openai.embeddings.create(
    input="electronics",
    model="text-embedding-3-small"
)
search_vector = response.data[0].embedding

# 2. Pass the numerical search_vector to query_points
payload = qdrant_client.query_points(
    collection_name="Amazon_Electronics_Products",
    query=search_vector,               # Pass the list of floats here
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False                 # Note: in query_points it is with_vectors, not with_vector
)


In [18]:
payload.points

[ScoredPoint(id=1629, version=13, score=0.34347615, payload={'parent_asin': 'B003YD34PI'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1353, version=11, score=0.34145623, payload={'parent_asin': 'B07F8C1PMQ'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1867, version=15, score=0.333336, payload={'parent_asin': 'B01LX1Z17Z'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=322, version=3, score=0.33064395, payload={'parent_asin': 'B00HSX9ZB2'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=880, version=7, score=0.32894048, payload={'parent_asin': 'B00ALKJODS'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=746, version=6, score=0.3276618, payload={'parent_asin': 'B078TQMXKW'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1675, version=14, score=0.3255921, payload={'parent_asin': 'B000GAU7AM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1762, version=14, sco

In [19]:
len(payload.points)

1000

In [20]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [21]:
parent_asin_list

['B003YD34PI',
 'B07F8C1PMQ',
 'B01LX1Z17Z',
 'B00HSX9ZB2',
 'B00ALKJODS',
 'B078TQMXKW',
 'B000GAU7AM',
 'B011NAANHU',
 'B00SLKQRJO',
 'B01NBXUPMX',
 'B0B2RKFTPH',
 'B001DRULBM',
 'B09YLX1834',
 'B0038L1XYU',
 'B000E8VCKU',
 'B0BT1TM3S2',
 'B0829ZQ97G',
 'B00330U8UQ',
 'B09CL2VNPJ',
 'B00DKFJA2G',
 'B07RY1X43Z',
 'B07L2W17V6',
 'B001963Y9I',
 'B0B3SMDK54',
 'B083C4BLVQ',
 'B000OMQXF0',
 'B071NLL6Z3',
 'B0075X3F7K',
 'B007884QFW',
 'B00JVIPOC6',
 'B01CZFUOD0',
 'B088XCZM3B',
 'B00IEFS0A0',
 'B07ZWYLYFR',
 'B07CQVWJH1',
 'B00AB0YYJM',
 'B00OT7HSBO',
 'B07FN4CZTR',
 'B0B5Y4FVYY',
 'B00AZMP9BS',
 'B001JVPFV8',
 'B087RMRLNK',
 'B01M6CLLFN',
 'B00R17YGL4',
 'B00XHY9YTE',
 'B00HQ9DJS8',
 'B07HYG9FPT',
 'B01GYJJH5Y',
 'B00L7O2AGA',
 'B0983DRDTR',
 'B0BTQW3CBP',
 'B000ZSV4X4',
 'B083PQ4WDK',
 'B001RS5ASG',
 'B08T8BSHQ9',
 'B01443P08K',
 'B07GZNGX4F',
 'B075P7WPJT',
 'B07H2Z829Y',
 'B01DVL0GQ2',
 'B07Y1YBYV3',
 'B000Q6M5CY',
 'B08LMVFTL5',
 'B07QZYRRBR',
 'B01161BYG0',
 'B002RYV1T6',
 'B010ELNK

Load Amazon Electronics Dataset